# 01 | Data overview and business understanding

## Project purpose
Explore workforce composition and employee experience: hiring, retention, performance, engagement, satisfaction, attendance, pay, and work mode. This is observational data and cannot establish cause. Names and DOB are retained in the confidential raw source so Day 2 can demonstrate cleaning and date parsing; do not display identity fields or export names and exact DOBs.

## 1. Load source data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
df=pd.read_csv(Path("../data/raw/HRDataset_EDA_raw.csv")).rename(columns={'EmpID': 'employee_id', 'Employee_Name': 'employee_name', 'DOB': 'date_of_birth', 'Sex': 'sex', 'MaritalDesc': 'marital_status', 'Department': 'department', 'Position': 'job_title', 'RecruitmentSource': 'recruitment_source', 'DateofHire': 'hire_date', 'DateofTermination': 'termination_date', 'TermReason': 'termination_reason', 'EmploymentStatus': 'employment_status', 'Termd': 'terminated_source', 'Salary': 'salary', 'PerformanceScore': 'performance_score', 'EngagementSurvey': 'engagement_score', 'EmpSatisfaction': 'employee_satisfaction', 'Absences': 'absences', 'DaysLateLast30': 'days_late_last_30', 'WorkMode': 'work_mode'})
print("Rows and columns:",df.shape)
print("Rows are not previewed here to avoid exposing employee-level records; schema and aggregate summaries follow.")

## 2. Schema and statistical overview

In [ ]:
df.info()

In [ ]:
display(df.drop(columns=["employee_id","employee_name","date_of_birth","terminated_source"],errors="ignore").describe(include="all").T)

## 3. Cardinality and field types

In [ ]:
display(df.nunique(dropna=False).sort_values(ascending=False).rename("Unique values"))

In [ ]:
date_candidates=["date_of_birth","hire_date","termination_date"] # DOB is parsed in Day 2 and removed before export
category_candidates=["sex","marital_status","department","job_title","recruitment_source","termination_reason","employment_status","performance_score","work_mode"]
numeric_candidates=["salary","engagement_score","employee_satisfaction","absences","days_late_last_30"]
print("Date candidates:",date_candidates)
print("Categorical candidates:",category_candidates)
print("Numeric candidates:",numeric_candidates)

## 4. Missingness and duplicates

In [ ]:
missing=pd.DataFrame({"Missing":df.isna().sum(),"Percent":(100*df.isna().mean()).round(2)})
display(missing.sort_values("Missing",ascending=False))
missing.sort_values("Missing").tail(12)["Percent"].plot.barh(title="Missing values by field (%)",figsize=(8,5));plt.tight_layout();plt.show()
print("Exact duplicate rows:",df.duplicated().sum())
print("Repeated employee IDs:",df.employee_id.duplicated().sum())

## 5. Data quality issue log
| Issue | Evidence | Plan |
|---|---|---|
| Missing termination date | Often expected for active staff | Validate with status; preserve nulls |
| Other missingness | Survey, pay, attendance and categories | Measure by segments; avoid blanket fills |
| Mixed date formats | DOB, hire, and termination strings | Parse each documented format in Day 2; correct DOB century with hire-age logic |
| Numeric values stored as text | Salary and engagement | Convert to numeric companions |
| Duplicates / repeated IDs | Exact rows and repeated EmpID | Remove exact duplicates only |
| Negative absences / extreme values | Numeric plausibility concern | Flag and ask data owner |
| Sensitive fields | Employee_Name contains mixed case, extra spaces, and dangling middle initials | Normalize and split names in Day 2; never show or export name fields in cleaned analysis files |

## Business questions
1. How do satisfaction and engagement vary by department and work mode? 2. Do attendance patterns vary across employee segments? 3. How do recruitment channels and hiring trends vary over time? 4. What differs between active and terminated employees? Results are descriptive, not causal.